# Does the reward binning have any signal in it?

`goalward()` on the 3000-step sweep showed every model flat at P(step toward goal) ~0.32 for all 12 MODE
settings, against an exact `piR*` running 0.30 -> 0.70. The reason is the binning, not the optimisation.

Bins "split log R evenly", but `R = gamma**L`, so `log R = L log gamma` and the split is **uniform in arrival
time** -- 18 steps per bin at T=200, K=12. The maze's longest shortest-path is 20. So:

| bins | arrival L | can optimal play land here? |
| --- | --- | --- |
| 1-9 | 37-200 | no -- all mean "wandered" |
| 10 | 19-36 | yes |
| 11 | 0-18 | yes |

Nine of eleven bins request near-identical behaviour (exact `P(goalward)` moves only 0.30 -> 0.41 across bins
0..9) and together hold ~14% of the data. All the behavioural signal is in bins 10-11, ~8% of rollouts. A
model that ignores MODE is very nearly correct for 92% of its training data -- and that is what it learned.

`binning="geometric"` splits **log L** instead, concentrating resolution where optimal play lives. Measured,
bins usable by optimal play:

| K | uniform | geometric |
| --- | --- | --- |
| 6 | 1 of 5 | 3 of 5 |
| 12 | **2 of 11** | **7 of 11** |
| 24 | 3 of 23 | 11 of 23 |

Note uniform barely improves with more bins -- doubling K to 24 buys one extra usable bin -- so this is not a
resolution problem, it is a placement problem. Varying K here is to find where geometric starts to overreach:
at K=24 it leaves bins 18, 21 and 22 **empty** (no integer arrival time lands there), which are dead
value-head classes with `h = 0` everywhere and `piR*` undefined.

Only the outcome *labels* change. The stored rollouts are untouched, so no dataset rebuild -- each config
just needs its own exact test set (~0.6 s).

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

## 1. The configurations, and what they look like before any training

This cell is free and answers most of the question on its own: for each (binning, K) it reports how many bins
optimal play can reach, which bins are empty, how the data distributes, and the spread of the **exact**
conditioned policy across bins. If `piR*` is flat for a config, no model trained on it can do better.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from maze_consistency.dataset import canonical_maze, load as load_data
from maze_consistency.dp import compute_ground_truth
from maze_consistency.tokens import Tokenizer
from maze_consistency.testset import build_testset, load_testset, stratified_rows, score, FAR
from maze_consistency.train import train, load_run, RUNS_DIR, N_HELDOUT, LossConfig
from maze_consistency.model import ModelConfig, MazeTransformer, make_forward
from maze_consistency.evaluate import make_enrichment_eval
import jax.numpy as jnp

BINNINGS = ["uniform", "geometric"]   #@param
KS       = [6, 12, 24]                #@param
PREFIX   = "binning"

CONFIGS = {f"{b}_K{k}": dict(binning=b, n_bins=k) for b in BINNINGS for k in KS}
_, DATA = load_data()

def goalward_of(policy, closer, mask=None):
    pc = (policy * closer).sum(-1)
    return pc[mask].mean() if mask is not None else pc.mean()

print(f"{'config':<16}{'opt-usable':>11}{'empty':>14}{'data% top bins':>16}{'piR* P(goalward) lo->hi':>26}")
INFO = {}
for name, kw in CONFIGS.items():
    m = canonical_maze(**kw)
    gt = compute_ground_truth(m)
    cells = m.start_cells
    closer = np.stack([m.dist[m.next_open[cells, a]] < m.dist[cells] for a in range(4)], -1)
    ex = np.transpose(gt.piR_star[0, cells], (1, 0, 2))
    feas = gt.h[0, cells].T > 0
    g = np.array([goalward_of(np.nan_to_num(ex[k]), closer, feas[k]) if feas[k].any() else np.nan
                  for k in range(m.K)])
    cnt = np.bincount(m.outcome_bin(DATA["length"], DATA["reached"]), minlength=m.K)
    opt = sorted(set(int(m.success_bin(d)) for d in range(1, m.dist.max() + 1)))
    INFO[name] = dict(maze=m, gt=gt, goalward=g, counts=cnt, opt_bins=opt)
    print(f"{name:<16}{len(opt):>4} of {m.K-1:<4}{str(m.empty_bins or '-'):>14}"
          f"{100*cnt[opt].sum()/cnt.sum():>15.1f}%{np.nanmin(g):>14.2f} ->{np.nanmax(g):>6.2f}")

fig, ax = plt.subplots(1, len(KS), figsize=(4.6*len(KS), 3.8), squeeze=False)
for j, k in enumerate(KS):
    a = ax[0, j]
    for c, b in enumerate(BINNINGS):
        info = INFO[f"{b}_K{k}"]
        x = np.arange(info["maze"].K) / (info["maze"].K - 1)
        a.plot(x, info["goalward"], ".-", color=f"C{c}", label=f"{b} (exact piR*)")
    a.axhline(0.25, color="gray", lw=.8, ls=":")
    a.set_title(f"K = {k}", fontsize=9); a.set_xlabel("requested bin (normalised)")
    a.set_ylabel("exact P(step toward goal)"); a.grid(alpha=.3); a.legend(fontsize=7)
fig.tight_layout(); plt.show()

## 2. Build each config's exact test set

Only the labels differ, so this is just the DP plus exact rollouts -- under a second each. Cached next to the
runs so a reconnected session skips them.

In [ ]:
TESTSETS = {}
for name, kw in CONFIGS.items():
    path = os.path.join(RUNS_DIR, PREFIX, f"testset_{name}.npz")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        build_testset(INFO[name]["maze"], n_per=200, path=path, log=lambda *a: None)
    TESTSETS[name] = path
    print("ok", name, "->", path)

## 3. Train one model per config

`maze_kw` is forwarded to `dataset.load`, so the run relabels outcomes without touching the rollouts. Each
config gets its own tokenizer, model (the value head is K-wide) and test set, so **`act_kl` and `value_kl`
are NOT comparable across configs** -- different K means different targets. Section 4 handles that.

In [ ]:
def run_sweep(configs=None, seeds=(0,), steps=2000, batch=32, lr=1e-3, d_model=64, n_layers=2, n_heads=4,
              eval_every=250, eval_per_setting=50, loss=None, prefix=PREFIX, skip_existing=True, log=print):
    """One run per (config, seed). loss defaults to LossConfig(mc=True)."""
    done = {}
    for name, kw in (configs or CONFIGS).items():
        tok = Tokenizer(INFO[name]["maze"])
        ts = load_testset(TESTSETS[name])
        rows = stratified_rows(ts, eval_per_setting, seed=0)

        # the probe is per config: each binning has its own maze, bins and ceiling
        enrich = make_enrichment_eval(tok, INFO[name]["maze"], DATA, n=128)

        def eval_fn(params, fwd, tok=tok, ts=ts, rows=rows, enrich=enrich):
            m = score(params, fwd, tok, ts, rows)
            m.pop("per_setting")
            m.update(enrich(params, fwd))      # enrich/* and goalward/*
            return m

        for seed in seeds:
            run = f"{prefix}/{name}_s{seed}"
            if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
                log(f"[skip] {run} exists"); continue
            done[run] = train(name=run, steps=steps, batch=batch, lr=lr, d_model=d_model, n_layers=n_layers,
                              n_heads=n_heads, seed=seed, loss=loss or LossConfig(mc=True), eval_fn=eval_fn,
                              eval_every=eval_every, log=log, maze_kw=kw)
    return done


run_sweep(steps=2000)

## 4. The comparison that survives a change of bins

`act_kl` cannot be compared across configs, so ask each model the one question that means the same thing
under any binning: **condition on each start's own best bin and see how it plays.**

- `P(goalward | best)` -- probability mass on a step toward the goal. 0.25 = no preference.
- `KL_opt | best` -- KL(optimal policy || model). Lower is better here, unambiguously, because the request
  *is* "play optimally".
- the `piR*` row is the ceiling this data can teach under that binning.

`maze.best_bin(s) = success_bin(dist[s])` is well defined under either scheme, which is what makes the row
comparable.

In [ ]:
def best_bin_play(prefix=PREFIX, seed=0, far=FAR):
    rows = {}
    for name in CONFIGS:
        m, gt = INFO[name]["maze"], INFO[name]["gt"]
        tok = Tokenizer(m)
        cells = m.start_cells
        closer = np.stack([m.dist[m.next_open[cells, a]] < m.dist[cells] for a in range(4)], -1)
        pi_opt = closer / closer.sum(-1, keepdims=True)
        kbest = m.best_bin(cells)
        is_far = m.dist[cells] >= far

        def stats(q):
            kl = (pi_opt * (np.log(np.maximum(pi_opt, 1e-30)) - np.log(np.maximum(q, 1e-30)))).sum(-1)
            pc = (q * closer).sum(-1)
            return pc.mean(), pc[is_far].mean(), kl.mean(), kl[is_far].mean()

        ex = gt.piR_star[0, cells, kbest]                      # exact policy at each start's own best bin
        rows[(name, "piR*")] = stats(np.nan_to_num(ex, nan=0.25))
        try:
            params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
        except FileNotFoundError:
            continue
        fwd = make_forward(MazeTransformer(mcfg), tok)
        x = tok.blank(len(cells)); x[:, 0] = tok.mode(kbest); x[:, 1] = tok.pos(cells)
        lg = np.asarray(fwd(params, jnp.asarray(x))["pi_logits"][:, 0]).astype(np.float64)
        q = np.exp(lg - lg.max(-1, keepdims=True)); q /= q.sum(-1, keepdims=True)
        rows[(name, "model")] = stats(q)

    hdr = ["P(goalward)", "  far", "KL_opt", "  far"]
    print(f"{'config':<16}{'who':<7}" + "".join(f"{h:>13}" for h in hdr))
    for name in CONFIGS:
        for who in ("model", "piR*"):
            if (name, who) in rows:
                print(f"{name:<16}{who:<7}" + "".join(f"{v:13.3f}" for v in rows[(name, who)]))
        print()

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    names = [n for n in CONFIGS if (n, "model") in rows]
    xs = np.arange(len(names))
    for i, (col, lab) in enumerate([(0, "P(step toward goal) | best bin"), (2, "KL(optimal || model) | best bin")]):
        ax[i].bar(xs - .2, [rows[(n, "model")][col] for n in names], .4, label="model")
        ax[i].bar(xs + .2, [rows[(n, "piR*")][col] for n in names], .4, label="piR* (ceiling)")
        ax[i].set_xticks(xs); ax[i].set_xticklabels(names, rotation=40, ha="right", fontsize=8)
        ax[i].set_title(lab, fontsize=9); ax[i].grid(alpha=.3, axis="y"); ax[i].legend(fontsize=8)
    ax[0].axhline(0.25, color="gray", lw=.8, ls=":")
    fig.tight_layout()
    return rows


_ = best_bin_play()
plt.show()

## 5. The full response curve per config

Same statistic as `goalward()` in the other notebooks, one panel per config. The x axis is the bin index
normalised to [0, 1] so curves with different K line up. What you want to see: the model tracking `piR*`
upward instead of sitting flat.

In [ ]:
def response_curves(prefix=PREFIX, seed=0):
    fig, ax = plt.subplots(len(BINNINGS), len(KS), figsize=(4.4*len(KS), 3.6*len(BINNINGS)), squeeze=False)
    for i, b in enumerate(BINNINGS):
        for j, k in enumerate(KS):
            name = f"{b}_K{k}"
            a, m, gt = ax[i, j], INFO[name]["maze"], INFO[name]["gt"]
            tok = Tokenizer(m); cells = m.start_cells
            closer = np.stack([m.dist[m.next_open[cells, a_]] < m.dist[cells] for a_ in range(4)], -1)
            xs = np.arange(m.K) / (m.K - 1)
            a.plot(xs, INFO[name]["goalward"], "k.--", lw=1.4, label="piR* (exact)")
            try:
                params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
                fwd = make_forward(MazeTransformer(mcfg), tok)
                g = []
                for kk in range(m.K):
                    x = tok.blank(len(cells)); x[:, 0] = tok.mode(np.full(len(cells), kk)); x[:, 1] = tok.pos(cells)
                    lg = np.asarray(fwd(params, jnp.asarray(x))["pi_logits"][:, 0]).astype(np.float64)
                    q = np.exp(lg - lg.max(-1, keepdims=True)); q /= q.sum(-1, keepdims=True)
                    g.append((q * closer).sum(-1).mean())
                a.plot(xs, g, ".-", color="C0", label="model")
            except FileNotFoundError:
                pass
            for e in m.empty_bins:
                a.axvline(e / (m.K - 1), color="r", lw=.8, alpha=.4)
            a.axhline(0.25, color="gray", lw=.8, ls=":")
            a.set_title(f"{name}  (empty bins: {m.empty_bins or 'none'})", fontsize=8)
            a.set_xlabel("requested bin (normalised)"); a.grid(alpha=.3); a.legend(fontsize=7)
    fig.tight_layout()
    return fig


response_curves()
plt.show()

## 6. Enrichment: does a better binning actually buy conditioning?

`best_bin_play` above compares each model to `piR*` at t = 0. This is the paired version over every prefix,
and it is the number that decides the experiment: at the identical prefix, ask for bin 0 versus the best
outcome still achievable, and measure the gain in goalward probability mass.

It is **binning-invariant in points** (goalward mass is goalward mass), while `frac_of_exact` is normalised
by each config's own ceiling -- use points to compare configs directly, and the fraction to ask how much of
the available signal each config's model actually used.

If geometric bins help, this is where it shows: a larger ceiling *and* a larger model gain. A larger ceiling
with an unchanged model gain means the extra signal went unused.

In [ ]:
def enrichment_by_config(prefix=PREFIX, seed=0, n=256):
    rows = {}
    for name in CONFIGS:
        m = INFO[name]["maze"]
        probe = make_enrichment_eval(Tokenizer(m), m, DATA, n=n)
        try:
            params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
        except FileNotFoundError:
            rows[name] = (np.nan, np.nan, np.nan, probe.ceiling); continue
        r = probe(params, make_forward(MazeTransformer(mcfg), Tokenizer(m)))
        rows[name] = (r["goalward/fail"], r["goalward/best"], r["enrich/points"], probe.ceiling)

    w = max(len(k) for k in rows) + 2
    print(f"{'config':<{w}}{'ask: fail':>11}{'ask: best':>11}{'model gain':>12}{'ceiling':>10}{'% used':>9}")
    for name, (f, b, d, ceil) in rows.items():
        pct = f"{100*d/ceil:8.0f}%" if np.isfinite(d) and ceil else f"{'-':>9}"
        print(f"{name:<{w}}{f:10.1f}%{b:10.1f}%{d:+11.1f}{ceil:+10.1f}{pct}")

    xs = np.arange(len(rows))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].bar(xs - .2, [r[2] for r in rows.values()], .4, label="model gain")
    ax[0].bar(xs + .2, [r[3] for r in rows.values()], .4, label="ceiling (exact)")
    ax[0].set_ylabel("points of goalward mass"); ax[0].set_title("enrichment", fontsize=9)
    ax[1].bar(xs, [100*r[2]/r[3] if np.isfinite(r[2]) and r[3] else np.nan for r in rows.values()], .5)
    ax[1].set_ylabel("% of that config's ceiling"); ax[1].set_title("share of available signal used", fontsize=9)
    for a in ax:
        a.set_xticks(xs); a.set_xticklabels(list(rows), rotation=40, ha="right", fontsize=8); a.grid(alpha=.3, axis="y")
    ax[0].legend(fontsize=8)
    fig.tight_layout()
    return rows


_ = enrichment_by_config()
plt.show()

## Hacking this

- **Other K**: edit `KS` and re-run from cell 1. Test sets rebuild automatically for new configs.
- **Add the consistency term**: `run_sweep(loss=LossConfig(mc=True, cons=True, cons_loss="local", w_cons=0.15))`
  with a fresh `prefix`. Worth doing *after* picking a binning -- on the uniform bins there is barely a signal
  for it to propagate.
- **A different scheme entirely**: `Maze.outcome_bin` is the only place the partition lives, plus `L_edges`
  for display. Binning by excess over optimal (`L - dist(start)`) would be start-independent and is the more
  principled fix, but the DP's `(t, s)` state cannot express it -- it would need wasted-steps-so-far as state.
- **Careful**: `act_kl` / `value_kl` are per-binning and never comparable across configs. Cross-config claims
  belong in section 4, which is binning-invariant by construction.